# Bentopy Tutorial 2 — Packing Around an Existing Structure

This notebook follows the [Bentopy tutorial](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)
section *Tutorial 2: Packing Around Existing Structures*.

We add a single POPC bilayer as **excluded volume**, place 300 lysozymes
in the bulk solvent, and 100 ubiquitins **within 5 nm of the membrane
surface** using a proximity rule.

New tools introduced:

* **`bentopy-mask`** — convert a structure into a voxel mask
* **`bentopy-merge`** — concatenate `.gro` files
* The `combines` and `around` rules in `.bent` files


## 0. Verify the workspace

This notebook is meant to be opened from
`/projects/bgvl/$USER/Martini/tutorial_2/` (created earlier by
`setup_martini.sh`).  All outputs of the cells below land in that folder.


In [ ]:
import os, pathlib, shutil, sys

# Make the bentopy CLI tools available no matter which Python kernel the
# user picks. The shared venv lives in the workshop's allocation.
VENV_BIN = "/projects/bgvl/alfiaparvez/bentopy_tutorial/.venv/bin"
if VENV_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = VENV_BIN + ":" + os.environ["PATH"]

# This notebook is meant to be opened from inside its tutorial folder,
# e.g.  /projects/bgvl/$USER/SummerSchool_2026/Martini/tutorial_2/ .  Outputs of
# every cell below (.bent, .json, .gro, .top, ...) land in this folder.
NB_DIR = pathlib.Path.cwd()

missing = [d for d in ("structures", "topology", "mdp_files")
           if not (NB_DIR / d).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing input directories in {NB_DIR}: {missing}.\n"
        f"Run  bash Martini/copy.sh  first, then open the notebook from "
        f"/projects/bgvl/$USER/SummerSchool_2026/Martini/tutorial_2/."
    )

print("Working dir :", NB_DIR)
print("bentopy-pack:", shutil.which("bentopy-pack"))


## 1. Inspect the compartments identified in the membrane

`bentopy-mask --visualize-labels` writes a `.gro` file with one bead per
voxel, named after its compartment. Negative labels (`-1`, `-2`, …) are
the open / solvent regions, positive labels (`1`, `2`, …) the excluded
solid structures.


In [ ]:
!bentopy-mask structures/membrane.gro -b labels.gro
print('  labels.gro size:', pathlib.Path('labels.gro').stat().st_size, 'bytes')


You can open `labels.gro` in VMD and use the selections

* `name "-1"` → the outside solvent region (available for packing)
* `name 1`    → the membrane region (excluded)


## 2. Create the membrane mask

We turn label `1` (the membrane voxels) into a `.npz` mask that the next
step will load.


In [ ]:
!bentopy-mask structures/membrane.gro -l 1:membrane_mask.npz


## 3. Write the new recipe `membrane_packing.bent`

Three new ideas:

* `membrane from "membrane_mask.npz"` — load a compartment from a mask file.
* `solvent combines not membrane` — boolean combination of compartments.
* `close-to-membrane around 5 of membrane` — proximity rule (5 nm shell).

The `:lyz` / `:ubq` suffixes are *tags* that override the residue names in
the output `.gro`, which makes selecting them later in VMD trivial.


In [ ]:
bent = '''[ general ]
title "Proteins around a membrane"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/martini_v3.0.0_phospholipids_v1.itp"
"topology/lysozyme.itp"
"topology/ubiquitin.itp"

[ compartments ]
membrane           from "membrane_mask.npz"
solvent            combines not membrane
close-to-membrane  around 5 of membrane

[ segments ]
LYZ:lyz 300 from "structures/lysozyme.pdb"  in solvent
UBQ:ubq 100 from "structures/ubiquitin.pdb" in close-to-membrane
'''
pathlib.Path("membrane_packing.bent").write_text(bent)
print(bent)


## 4. Pack the system

Bentopy will pack the larger structure first (lysozyme is bigger than
ubiquitin under the default *moment* heuristic).


In [ ]:
!bentopy-pack membrane_packing.bent placements.json


## 5. Render and merge with the original membrane

`bentopy-render` writes only the packed proteins. `bentopy-merge`
concatenates the protein `.gro` with the original `membrane.gro` to
produce the full system. The lipid count is then appended to `topol.top`.


In [ ]:
!bentopy-render placements.json packed_proteins.gro -t topol.top
!bentopy-merge packed_proteins.gro structures/membrane.gro -o system.gro
with open('topol.top', 'a') as fh:
    fh.write('POPC    5408\n')
print(pathlib.Path('topol.top').read_text())


## 6. Solvation


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro -t topol.top \
    -s NA:0.15M -s CL:0.15M --charge neutral
print('--- final topol.top ---')
print(pathlib.Path('topol.top').read_text())


## 5. Visualize

Open the solvated system with VMD (run this in a Delta OnDemand desktop
session, not in the notebook itself):

```bash
export PATH=/projects/bgvl/alfiaparvez/software/vmd/bin:$PATH
cd /projects/bgvl/$USER/SummerSchool_2026/Martini/tutorial_2
vmd -e vis_t2.tcl solvated_system.gro
```

The QuickSurf rendering script `vis_t2.tcl` is in this folder already,
so VMD will pick it up directly.


### Expected result (Figure 5 of the tutorial)

* The membrane appears as an excluded slab of POPC.
* Lysozymes (`resname lyz`) are spread through the aqueous regions.
* Ubiquitins (`resname ubq`) cluster within 5 nm of the membrane surface.
* No protein overlaps with the membrane.
